## Execution record

Smaller local training run on 3 October 2026. Settings: {"LLM_MODEL_ID": "LiquidAI/LFM2-350M", "LLM_STEPS": "20", "LLM_CONTEXT": "256"}. Outputs below are from execution in this session. External services and audio playback may be skipped with an explicit reason.


# My language model adaptation notebook

Adapted from [MIT Introduction to Deep Learning](http://introtodeeplearning.com). I retain the course copyright and use its style dataset helpers. I explain the prompt format then the LoRA update and evaluation.

In [1]:
# Copyright 2026 MIT Introduction to Deep Learning. All Rights Reserved.
#
# Licensed under the MIT License. You may not use this file except in compliance
# with the License. Use and/or modification of this code outside of MIT Introduction
# to Deep Learning must reference:
#
# © MIT Introduction to Deep Learning
# http://introtodeeplearning.com
#

## Adapting an existing language model

I use a pretrained LiquidAI LFM2 model. The default has 1.2 billion parameters and LLM_MODEL_ID allows an explicit smaller variant for limited hardware. The base model is pretrained. My work here is the adaptation pipeline and the notes about what each stage does.

In [2]:
%pip install "setuptools<81" --quiet
%pip install mitdeeplearning --no-deps --no-build-isolation --quiet
import mitdeeplearning as mdl


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


C:\Users\Somil Singh\Documents\Codex\2026-10-03\new-chat\work\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
import os
import json
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
from torch.nn import functional as F
from torch.utils.data import DataLoader

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from lion_pytorch import Lion
torch.manual_seed(7)
np.random.seed(7)
torch.set_num_threads(4)
RUN_STEPS = int(os.getenv("LLM_STEPS", "200"))
print("Training steps per style:", RUN_STEPS)

print("Training context:", os.getenv("LLM_CONTEXT", "512"))


Training steps per style: 20
Training context: 256


## Learning a response style

I first practise a leprechaun style then adapt the model to the course Yoda style data. I keep style evaluation separate from whether an answer is factually correct.

## Chat formatting

The prompt marks the user turn and the start of the assistant turn. These control tokens are part of the format expected by the chosen model.

In [4]:
# Basic question answer template
template_without_answer = "<|startoftext|><|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n"
template_with_answer = template_without_answer + "{answer}<|im_end|>\n"

# Let's try to put something into the template to see how it looks
print(template_with_answer.format(question="What is your name?", answer="My name is Lili!"))


<|startoftext|><|im_start|>user
What is your name?<|im_end|>
<|im_start|>assistant
My name is Lili!<|im_end|>



## Tokenization

The tokenizer maps text into model token indices. Its vocabulary belongs to the pretrained model so I load the matching tokenizer.

In [5]:
# Load the tokenizer for Liquid AI LFM2-1.2B
model_id = os.getenv("LLM_MODEL_ID", "LiquidAI/LFM2-1.2B")
print("Selected model:", model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# How big is the tokenizer?
print(f"Vocab size: {len(tokenizer.get_vocab())}")


Selected model: LiquidAI/LFM2-350M


Vocab size: 64400


I check encoding and decoding on a short sentence. The decoded text should preserve the content apart from any intentional special token handling.

In [6]:
# Lets test out both steps:
text = "Here is some sample text!"
print(f"Original text: {text}")

# Tokenize the text
tokens = tokenizer.encode(text, return_tensors="pt")
print(f"Encoded tokens: {tokens}")

# Decode the tokens
decoded_text = tokenizer.decode(tokens[0], skip_special_tokens=True)
print(f"Decoded text: {decoded_text}")


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Original text: Here is some sample text!
Encoded tokens: tensor([[   1, 9151,  856, 1429, 6643, 3304,  510]])
Decoded text: Here is some sample text!


I combine the prompt template with the tokenizer before asking the model a question.

### My note on the two stages

The template supplies conversation structure and the tokenizer supplies numerical indices. Neither stage creates an answer by itself.

In [7]:
prompt = template_without_answer.format(question="What is the capital of France? Use one word.")
print(prompt)


<|startoftext|><|im_start|>user
What is the capital of France? Use one word.<|im_end|>
<|im_start|>assistant



The prompt ends at the assistant turn. Generation should continue from that position.

## Loading the base model

I use reduced precision where the hardware supports it. Model weights and training activations both use memory so loading successfully does not guarantee that training will fit.

In [8]:
model = AutoModelForCausalLM.from_pretrained(
    model_id, dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
print("Model device:", model.device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Loading weights:  21%|██        | 31/148 [00:00<00:00, 277.94it/s]


Loading weights:  40%|███▉      | 59/148 [00:00<00:00, 179.96it/s]


Loading weights:  54%|█████▍    | 80/148 [00:00<00:00, 183.00it/s]


Loading weights:  68%|██████▊   | 100/148 [00:00<00:00, 182.92it/s]


Loading weights:  82%|████████▏ | 121/148 [00:00<00:00, 190.13it/s]


Loading weights:  95%|█████████▌| 141/148 [00:00<00:00, 190.13it/s]


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 199.14it/s]


Model device: cuda:0


In [9]:
### Putting it together to prompt the model and generate a response ###

# 1. Construct the prompt in chat template form
question = "What is the capital of France? Use one word."
prompt = template_without_answer.format(question=question)

# 2. Tokenize the prompt
tokens = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

# 3. Feed through the model to predict the next token probabilities
with torch.no_grad():
    output = model(tokens)

    probs = F.softmax(output.logits, dim=-1)

# 4. Get the next token, according to the maximum probability
next_token = torch.argmax(probs[0, -1, :]).item()

# 5. Decode the next token
next_token_text = tokenizer.decode(next_token)

print(f"Prompt: {prompt}")
print(f"Predicted next token: {next_token_text}")


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


Prompt: <|startoftext|><|im_start|>user
What is the capital of France? Use one word.<|im_end|>
<|im_start|>assistant

Predicted next token: Paris


A forward pass predicts a distribution for the next token at each position. I inspect the final position because it predicts the continuation after the prompt.

### My note on decoding

Argmax selects the most likely token. Sampling can produce more varied responses. I keep those methods separate when comparing model behaviour.

In [10]:
prompt = template_without_answer.format(question="What does MIT stand for?")
tokens = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
output = model.generate(tokens, max_new_tokens=20)
print(tokenizer.decode(output[0]))


[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


<|startoftext|><|startoftext|><|im_start|>user
What does MIT stand for?<|im_end|>
<|im_start|>assistant
MIT stands for Massachusetts Institute of Technology. It is a public research university located in Cambridge, Massachusetts


The generation helper repeats next token prediction until it reaches a stopping condition or the requested limit.

## Loading style examples

The course loader supplies instructions with ordinary responses and responses written in the target style. I inspect one example before training.

In [11]:
from pathlib import Path

def create_style_dataloaders(style):
    dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
    style_path = Path(mdl.lab3.__file__).parent / "data" / "text_styles" / f"{style}.txt"
    # The course style files use UTF 8. I specify it so Windows reads every character.
    responses = [line.strip().replace("\\n", "\n") for line in style_path.read_text(encoding="utf-8").splitlines()]
    styled = dataset.select(range(len(responses))).map(
        lambda row, index: {"response_style": responses[index]}, with_indices=True
    )
    split = styled.train_test_split(test_size=0.2, seed=7)
    return DataLoader(split["train"], batch_size=1, shuffle=True), DataLoader(split["test"], batch_size=1)

train_loader, test_loader = create_style_dataloaders(style="leprechaun")

sample = train_loader.dataset[44]
question = sample['instruction']
answer = sample['response']
answer_style = sample['response_style']

print(f"Question: {question}\n\n" +
      f"Original Answer: {answer}\n\n" +
      f"Answer Style: {answer_style}")


Map:   0%|          | 0/2048 [00:00<?, ? examples/s]


Map: 100%|██████████| 2048/2048 [00:00<00:00, 19806.17 examples/s]


Question: What are some tools that every homeowner should have?

Original Answer: There is a tool designed for almost every challenge a homeowner will face, but purchasing every tool could prove to be cost prohibitive, especially if they are highly specialized for tasks that may occur infrequently.  It's best to have a general set of tools that have a wide variety of uses, including a hammer, a set of screwdrivers both flat and philips, a tape measure, a level, a stud finder, and a utility knife. A battery powered drill and impact driver can also save time, and make mounting projects easier.

Answer Style: Top o' the mornin' to ye! Now, when it comes to havin' a set of tools that'll see ye through the likes of home repairs and whatnot, ye needn't go buyin' every tool under the sun, fer that would put quite a dent in yer gold, wouldn't it? Instead, grab yerself a trusty hammer, a set o' screwdrivers both flat and Philips, a tape measure, a level, a stud finder, and a utility knife. Oh, 

## My chat helper

I wrap prompt preparation and generation in one function. The only_answer option removes the prompt tokens from the displayed result.

In [12]:
def chat(question, max_new_tokens=32, temperature=0.7, only_answer=False):
    # 1. Construct the prompt using the template
    prompt = template_without_answer.format(question=question)

    # 2. Tokenize the text
    input_ids = tokenizer(prompt, return_tensors="pt").to(model.device)

    # 3. Feed through the model to predict the next token probabilities
    with torch.no_grad():
        outputs = model.generate(**input_ids, do_sample=True, max_new_tokens=max_new_tokens, temperature=temperature)

    # 4. Only return the answer if only_answer is True
    output_tokens = outputs[0]
    if only_answer:
        output_tokens = output_tokens[input_ids['input_ids'].shape[1]:]

    # 5. Decode the tokens
    result = tokenizer.decode(output_tokens, skip_special_tokens=True)

    return result


I ask a simple question before training to establish the base behaviour. I use the same kinds of questions afterward for comparison.

In [13]:
# Let's try chatting with the model now to test if it works!
answer = chat(
    "What is the capital of Ireland?",
    only_answer=True,
    max_new_tokens=32,
)

print(answer)



The capital of Ireland is Dublin.


## LoRA adapters

I freeze the base model and train small low rank update matrices in selected projection layers. The printed counts show the fraction of parameters that can change.

In [14]:
# LoRA is a way to finetune LLMs very efficiently by only updating a small subset of the model's parameters

def apply_lora(model):
    if hasattr(model, "peft_config"):
        return model
    # Define LoRA config
    lora_config = LoraConfig(
        r=8, # rank of the LoRA matrices
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"
        ],
    )

    # Apply LoRA to the model
    lora_model = get_peft_model(model, lora_config)
    return lora_model

model = apply_lora(model)

# Print the number of trainable parameters after applying LoRA
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"number of trainable parameters: {trainable_params}")
print(f"total parameters: {total_params}")
print(f"percentage of trainable parameters: {trainable_params / total_params * 100:.2f}%")

# note to self: r=8 keeps the adapter tiny (usually <1% of total params trainable). that's the whole
# point of LoRA, we freeze the base weights and only learn these small low rank update matrices per layer


number of trainable parameters: 245760
total parameters: 354729728
percentage of trainable parameters: 0.07%


## Computing the answer loss

I shift the token sequence by one position for next token prediction. I mask prompt tokens so the objective concentrates on the answer.

In [15]:
def forward_and_compute_loss(model, tokens, mask, context_length=int(os.getenv("LLM_CONTEXT", "512"))):
    # note to self: x is tokens[:-1] and y is tokens[1:] because this is next token prediction --
    # at every position the model is trained to predict the single token that comes right after it
    # Truncate to context length
    tokens = tokens[:, :context_length]
    mask = mask[:, :context_length]

    # Construct the input, output and mask
    x = tokens[:, :-1]
    y = tokens[:, 1:]
    mask = mask[:, 1:]

    # Forward pass to compute logits
    logits = model(x).logits

    # Compute loss
    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        y.reshape(-1),
        reduction="none"
    )

    # Mask out the loss for non answer tokens
    selected = mask.reshape(-1).bool()
    if not selected.any():
        raise ValueError("No answer tokens remain after truncation")
    loss = loss[selected].mean()

    return loss


## My training loop

I attach adapters only once and clear gradients for each update. I check that the truncated sequence still includes answer tokens before averaging their loss.

In [16]:
### Training loop ###

def train(model, dataloader, tokenizer, max_steps=200, context_length=int(os.getenv("LLM_CONTEXT", "512")), learning_rate=1e-4):
    losses = []
    updates = 0

    # Apply LoRA to the model
    model = apply_lora(model)
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    model.config.use_cache = False
    model.train()

    optimizer = Lion(model.parameters(), lr=learning_rate)

    # Training loop
    for step, batch in enumerate(dataloader):
        question = batch["instruction"][0]
        answer = batch["response_style"][0]

        # Format the question and answer into the template
        text = template_with_answer.format(question=question, answer=answer)

        # Tokenize the text and compute the mask for the answer
        ids = tokenizer(text, return_tensors="pt", return_offsets_mapping=True).to(model.device)
        answer_start = len(template_without_answer.format(question=question))
        mask = ids["offset_mapping"][:,:,1] > answer_start
        # note to self: this mask is what makes it "loss on the answer only". we don't want to
        # penalize the model for not predicting the question tokens, only for getting the answer right

        # Feed the tokens through the model and compute the loss
        if not mask[:, 1:context_length].any():
            continue
        loss = forward_and_compute_loss(
            model=model,
            tokens=ids["input_ids"],
            mask=mask,
            context_length=context_length,
        )

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        updates += 1

        losses.append(loss.item())

        # monitor progress
        if step % 10 == 0:
            model.eval()
            print(chat("What is the capital of France?", only_answer=True))
            model.train()
            print(f"step {step} loss: {torch.mean(torch.tensor(losses)).item()}")
            losses = []

        if updates >= max_steps:
            break

    print(f"Completed adapter updates: {updates}")
    model.eval()
    model.config.use_cache = True
    return model


In [17]:
# Call the train function to fine tune the model! Hint: you'll start to see results after at least 100 steps.
model = train(model, train_loader, tokenizer, max_steps=RUN_STEPS)


The capital of France is Paris. It is renowned for its iconic landmarks such as the Eiffel Tower, Notre Dame Cathedral, and the Louvre Museum.
step 0 loss: 2.84765625


The capital of France is Paris. Paris is a city in the north-central part of the country and is known for its historical landmarks like the Eiffel
step 10 loss: 3.215039014816284


Completed adapter updates: 20


I generate another response after training. A change in wording is useful evidence but one response is not a complete evaluation.

### My note on measurement

Style can be assessed with examples and a scoring rubric. A judge model adds another imperfect measurement so I would inspect its decisions as well as its average score.

In [18]:
print(chat("What is a good story about tennis", only_answer=True, max_new_tokens=200))


Certainly! Here’s a captivating story set in the world of tennis, blending historical intrigue, personal drama, and the electrifying spirit of the game:

### **Title: The Serve of Dreams**

#### **Setting:**
In the late 19th century, in the picturesque town of Saint-Étienne, France, the world of tennis was a stage for both the elite and the rising stars. The town, nestled between rolling hills and lush vineyards, was a place where tradition and innovation collided.

#### **Characters:**
- **Lena:** A young, ambitious young player at just 18, known for her quick footwork and a mysterious past that she keeps hidden.
- **Major Marcel:** A rising star in the local tennis club, known for his aggressive serve and relentless spirit. He's a bit of an enigma, with a reputation for being as unpredictable as the game itself.
- **Coach Étienne


## Evaluating style

I compare ordinary answers with generated answers and target style examples. I keep the judge rubric fixed for that comparison.

## Adapting to a second style

This cell continues training the same adapter on Yoda style examples. It is a sequential adaptation experiment. A clean comparison would reload the base model and start a fresh adapter.

In [19]:
# Load the Yoda speak dataset and fine tune the model using your training function
train_loader, test_loader = create_style_dataloaders(style="yoda")
model = train(model, train_loader, tokenizer, max_steps=RUN_STEPS)


Map:   0%|          | 0/2048 [00:00<?, ? examples/s]


Map: 100%|██████████| 2048/2048 [00:00<00:00, 24664.30 examples/s]


The capital of France is Paris.
step 0 loss: 3.693359375


The capital of France is Paris. It is a major city known for its historical landmarks, art, fashion, gastronomy, and culture.
step 10 loss: 3.399707078933716


Completed adapter updates: 20


I give the judge a style example and a scoring scale. I request a small JSON response so the result can be checked programmatically.

In [20]:
### LLM as a judge ###



system_prompt = """
You are an impartial judge that evaluates if text was written by {style}.

An example piece of text from {style} is:
{example}

Now, analyze some new text carefully and respond on if it follows the
same style of {style}. Be critical to identify any issues in the text.
Then convert your feedback into a number between 0 and 10: 10 if the text
is written exactly in the style of {style}, 5 if mixed faithfulness to the
style, or 0 if the text is not at all written in the style of {style}.

Directly answer with the score formatted in a dictionary.
The format of your response should only be the dictionary and nothing else:
{{"score": <score between 0 and 10>}}
"""

style = "Yoda"
example = "The very Republic is threatened, if involved the Sith are. Hard to see, the dark side is. Discover who this assassin is, we must. With this Naboo queen you must stay, Qui-Gon. Protect her. May the Force be with you. A vergence, you say? But you do! Revealed your opinion is. Trained as a Jedi, you request for him? Good, good, young one."

system_prompt = system_prompt.format(style=style, example=example)
print("=== System prompt ===")
print(system_prompt)


=== System prompt ===

You are an impartial judge that evaluates if text was written by Yoda.

An example piece of text from Yoda is:
The very Republic is threatened, if involved the Sith are. Hard to see, the dark side is. Discover who this assassin is, we must. With this Naboo queen you must stay, Qui-Gon. Protect her. May the Force be with you. A vergence, you say? But you do! Revealed your opinion is. Trained as a Jedi, you request for him? Good, good, young one.

Now, analyze some new text carefully and respond on if it follows the
same style of Yoda. Be critical to identify any issues in the text.
Then convert your feedback into a number between 0 and 10: 10 if the text
is written exactly in the style of Yoda, 5 if mixed faithfulness to the
style, or 0 if the text is not at all written in the style of Yoda.

Directly answer with the score formatted in a dictionary.
The format of your response should only be the dictionary and nothing else:
{"score": <score between 0 and 10>}



## Optional judge service

The judge uses an OpenRouter account and may incur usage charges. I read its key from the environment. Without a key I leave this stage unexecuted and state that no judge scores were produced.

In [21]:
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")
model_name = "google/gemini-2.5-flash"
llm = mdl.lab3.LLMClient(model=model_name, api_key=OPENROUTER_API_KEY) if OPENROUTER_API_KEY else None
print("Judge service configured:", llm is not None)


Judge service configured: False


## My evaluation metric

I parse the returned score and normalise it to the interval from zero to one. I reject missing or nonnumeric values rather than treating them as valid scores.

Opik supplies the metric interface. I use it to keep the scoring result consistent with the course evaluation helpers.

In [22]:
from opik.evaluation.metrics import base_metric, score_result

class LLMJudgeEvaluator(base_metric.BaseMetric):
    def __init__(self, judge: mdl.lab3.LLMClient = None, system_prompt: str = None):
        self.judge = judge
        self.system_prompt = system_prompt
        self.prompt_template = "Evaluate this text: {text}"

    def score(self, text: str, n_tries=20, **kwargs):
        """ Evaluate by asking an LLM to score it. """

        for attempt in range(n_tries):
            try:
                prompt = self.prompt_template.format(text=text)

                res = self.judge.ask(
                    system=self.system_prompt,
                    user=prompt,
                    max_tokens=20,
                )
                # res = self.judge.ask(

                # Extract the assistant's content from the API response
                res = res.choices[0].message.content
                res_dict = json.loads(res)

                max_score = 10 # The maximum score that the LLM should output
                raw_score = res_dict["score"]
                if isinstance(raw_score, bool) or not isinstance(raw_score, (int, float)):
                    raise ValueError("The judge must return a numeric score")
                score = raw_score / max_score # Normalize
                score = max(0.0, min(score, 1.0)) # Clip between 0 and 1

                # Return the score object
                return score_result.ScoreResult(name="StyleScore", value=score)

            except Exception as e:
                if attempt == n_tries - 1:  # Last attempt
                    raise e  # Re-raise the exception if all attempts failed
                continue  # Try again if not the last attempt


I create the evaluator only when the judge client is available.

In [23]:
judge = LLMJudgeEvaluator(llm, system_prompt=system_prompt) if llm else None


## Checking the rubric

I try a few short examples to see whether the judge distinguishes ordinary wording from the target style.

In [24]:
def scoring_function(text):
    return judge.score(text).value

test_texts = [
    "Tennis is a fun sport. But you must concentrate.",
    "Fun sport, tennis is. But work hard, you must.",
    "Hard to see, the dark side is."
]

for text in test_texts if judge else []:
    score = scoring_function(text)
    print(f"{text} ==> Score: {score}")

if judge is None:
    print("Judge scoring skipped: OPENROUTER_API_KEY is not configured.")


Judge scoring skipped: OPENROUTER_API_KEY is not configured.


I generate answers to held out instructions and compare them with control texts. The sample count is printed in the saved output.

In [25]:
# Generate text from your model by asking it new questions.
def generate_samples_from_test(test_loader, num_samples):
    samples = []
    for test_sample in tqdm(test_loader, total=num_samples):
        test_question = test_sample['instruction'][0]
        with torch.no_grad():
            generated = chat(test_question, only_answer=True, max_new_tokens=100)
        samples.append(generated)
        if len(samples) >= num_samples:
            break
    return samples

n_samples = 20
generated_samples = generate_samples_from_test(test_loader, num_samples=n_samples)


  0%|          | 0/20 [00:00<?, ?it/s]


  5%|▌         | 1/20 [00:02<00:47,  2.51s/it]


 10%|█         | 2/20 [00:05<00:53,  2.99s/it]


 15%|█▌        | 3/20 [00:09<00:55,  3.26s/it]


 20%|██        | 4/20 [00:13<00:54,  3.39s/it]


...

 80%|████████  | 16/20 [00:49<00:13,  3.30s/it]


 85%|████████▌ | 17/20 [00:53<00:10,  3.38s/it]


 90%|█████████ | 18/20 [00:57<00:06,  3.45s/it]


 95%|█████████▌| 19/20 [01:00<00:03,  3.48s/it]


 95%|█████████▌| 19/20 [01:04<00:03,  3.37s/it]


I collect ordinary and target style responses from the test split so the comparison does not rely on training examples.

In [26]:
import itertools
controls = list(itertools.islice(test_loader, n_samples))
base_samples = [sample['response'][0] for sample in controls]
style_samples = [sample['response_style'][0] for sample in controls]


## Scoring the samples

I score sequentially in the notebook. This avoids process spawning problems on Windows and makes API usage easier to inspect.

In [27]:
def compute_scores_in_parallel(samples):
    return [scoring_function(sample) for sample in samples] if judge else []
base_scores = compute_scores_in_parallel(base_samples)
generated_scores = compute_scores_in_parallel(generated_samples)
style_scores = compute_scores_in_parallel(style_samples)
if judge:
    for name, values in [("Base", base_scores), ("Generated", generated_scores), ("Style", style_scores)]:
        print(f"{name}: {np.mean(values):.2f} ± {np.std(values):.2f}")
else:
    print("No external judge scores were produced.")


No external judge scores were produced.


I compare both the average scores and their spread. A mean alone can hide inconsistent responses.

In [28]:
import seaborn as sns
import pandas as pd
if judge:
    df = pd.DataFrame({
        'Score': [*base_scores, *generated_scores, *style_scores],
        'Type': ['Base'] * len(base_scores) + ['Generated'] * len(generated_scores) + ['Style'] * len(style_scores)
    })
    sns.histplot(data=df, x='Score', hue='Type', multiple="dodge", bins=6, shrink=.8)
    plt.title('Distribution of style scores')
    plt.show()
else:
    print("Score plot skipped because no judge scores exist.")


Score plot skipped because no judge scores exist.


### My note on the controls

Ordinary responses are the negative control and the target style examples are the positive control. Generated responses should be interpreted relative to both.

I would review failures before changing the prompt or training settings. The judge can reward superficial phrasing so I also inspect whether answers remain useful.

## Optional tracing

Tracing records prompts and responses in an external service. I enable it only when an Opik key is configured.

In [29]:
import opik
from opik import opik_context
OPIK_API_KEY = os.getenv("OPIK_API_KEY", "")
tracing_enabled = bool(OPIK_API_KEY and judge)
if tracing_enabled:
    os.environ["OPIK_PROJECT_NAME"] = "My_LLM_Style_Experiment"
    opik.configure(use_local=False)
print("External tracing enabled:", tracing_enabled)


External tracing enabled: False


A trace can help connect a response with its evaluation. The service receives the logged content so I keep this stage explicit.

In [30]:
def inference_chat(question, max_new_tokens=32, temperature=0.7, only_answer=False):

    # 1. Construct the prompt using the template
    prompt = template_without_answer.format(question=question)

    # 2. Tokenize the text
    input_ids = tokenizer(prompt, return_tensors="pt").to(model.device)

    # 3. Feed through the model to predict the next token probabilities
    with torch.no_grad():
        outputs = model.generate(**input_ids, do_sample=True, max_new_tokens=max_new_tokens, temperature=temperature)

    # 4. Only return the answer if only_answer is True
    output_tokens = outputs[0]
    if only_answer:
        output_tokens = output_tokens[input_ids['input_ids'].shape[1]:]

    # 5. Decode the tokens
    result = tokenizer.decode(output_tokens, skip_special_tokens=True)

    # Update the current trace with evaluation scores
    opik_context.update_current_trace(
        feedback_scores=[
            {
                "name": "Yoda style eval",
                "value": scoring_function(result)
            }
        ]
    )

    return result

if tracing_enabled:
    inference_chat = opik.track(inference_chat)


When tracing is enabled the next call records an example response and its style score. Without credentials it prints a clear skip message.

### My note on monitoring

A style score is one signal. I would also track factual errors and response usefulness when evaluating a real assistant.

In [31]:
if tracing_enabled:
    # Let's try chatting with the model now to see the traces produced with the score
    answer = inference_chat(
        "Who was the only non-Jedi to wield a lightsaber in the original Star Wars trilogy?",
        only_answer=True,
        max_new_tokens=32,
    )
    
    print(answer)
else:
    print("Tracing example skipped because the services are not configured.")


Tracing example skipped because the services are not configured.


## Checking held out likelihood

The final local metric measures next token cross entropy on the course Yoda text. The printed value is a loss so lower is better. It does not replace a style or factuality assessment.

In [32]:
# DO NOT CHANGE/MODIFY THIS CELL.
# EXECUTE IT BEFORE SUBMITTING YOUR ENTRY TO THE LAB.

yoda_test_text = mdl.lab3.yoda_test_text
tokens = tokenizer(yoda_test_text, return_tensors="pt").to(model.device)

# Get the loglikelihood from the model
with torch.no_grad():
    outputs = model(**tokens)
    logits = outputs.logits[:, :-1]
    targets = tokens.input_ids[:, 1:]
    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                            targets.reshape(-1))

print(f"Yoda test loglikelihood: {loss.item():.2f}")


Yoda test loglikelihood: 3.97


## What I would investigate next

I would run fresh adapters for each style and repeat the experiment with fixed test prompts. I would report the run length and hardware alongside every result so a smaller local run is not confused with the full experiment.